In [60]:
import pandas as pd
from glob import glob
import numpy as np
import os
import json

In [61]:
root = "/media/yesindeed/DATADRIVE1/mount/remote_cse/datasets/OmniMedVQA"

data_all = []

for dataset_file in glob(os.path.join(root, "QA_information", "Open-access", "*.json")):
    dataset = os.path.basename(dataset_file)[:-5]

    with open(dataset_file) as fp:
        data = json.load(fp)

    data_all.extend(data)

data_all

[{'dataset': 'ACRIMA',
  'question_id': 'ACRIMA_0000',
  'question_type': 'Modality Recognition',
  'question': 'What imaging technique was employed to obtain this picture?',
  'gt_answer': 'Fundus imaging',
  'image_path': 'Images/ACRIMA/Im553_g_ACRIMA.png',
  'option_A': 'PET scan',
  'option_B': 'CT scan',
  'option_C': 'Blood test',
  'option_D': 'Fundus imaging',
  'modality_type': 'Fundus Photography'},
 {'dataset': 'ACRIMA',
  'question_id': 'ACRIMA_0001',
  'question_type': 'Modality Recognition',
  'question': 'What modality was used to capture this image?',
  'gt_answer': 'Fundus imaging',
  'image_path': 'Images/ACRIMA/Im351_g_ACRIMA.png',
  'option_A': 'Biopsy',
  'option_B': 'CT scan',
  'option_C': 'Colonoscopy',
  'option_D': 'Fundus imaging',
  'modality_type': 'Fundus Photography'},
 {'dataset': 'ACRIMA',
  'question_id': 'ACRIMA_0002',
  'question_type': 'Modality Recognition',
  'question': 'What method was employed to obtain this picture?',
  'gt_answer': 'Fundus im

In [62]:
unique_image_path_l = np.unique([x["image_path"] for x in data_all])
len(unique_image_path_l)

82059

In [63]:
len(data_all)

88996

In [64]:
np.random.shuffle(unique_image_path_l)
# sample 10%
unique_image_path_l = unique_image_path_l[: int(len(unique_image_path_l) * 0.1)]

train_image_l = unique_image_path_l[: int(len(unique_image_path_l) * 0.3)]
test_image_l = unique_image_path_l[int(len(unique_image_path_l) * 0.3) :]

In [65]:
len(unique_image_path_l)

8205

In [66]:
from tqdm import tqdm

datasets = np.unique([x["dataset"] for x in data_all])
split_summary = {}
for dataset in datasets:
    split_summary[dataset] = {"question_train": 0, "question_test": 0}

train_json_l = []
test_json_l = []

for item in tqdm(data_all):
    item_ = {
        "question_id": item["question_id"],
        "dataset": item["dataset"],
        "image_path": item["image_path"],
        "access_type": "Open-access",
    }

    if item["image_path"] in train_image_l:
        train_json_l.append(item_)
        split_summary[item["dataset"]]["question_train"] += 1
    elif item["image_path"] in test_image_l:
        test_json_l.append(item_)
        split_summary[item["dataset"]]["question_test"] += 1

df_split_summary = pd.DataFrame(split_summary)
df_split_summary

  1%|▏         | 1305/88996 [00:00<00:13, 6532.88it/s]

100%|██████████| 88996/88996 [00:22<00:00, 4011.80it/s]


,ACRIMA,ALL Challenge,Adam Challenge,BioMediTech,Blood Cell,BreakHis,COVIDx CXR-4,CRC100k,Chest CT Scan,Chest X-Ray PA,...,OLIVES,PAD-UFES-20,PALM2019,Pulmonary Chest MC,Pulmonary Chest Shenzhen,RUS CHN,RadImageNet,Retinal OCT-C8,SARS-CoV-2 CT-scan,Yangxi
question_train,10,5,2,18,27,21,16,54,26,26,...,22,17,18,0,1,58,1691,140,29,46
question_test,10,32,10,38,64,52,34,84,54,55,...,47,32,28,1,17,141,3925,273,59,110


In [67]:
df_split_summary_ = df_split_summary.T
df_split_summary_["dataset"] = df_split_summary_.index
df_split_summary_ = df_split_summary_.reset_index(drop=True)[["dataset", "question_train", "question_test"]]
df_split_summary_

,dataset,question_train,question_test
0,ACRIMA,10,10
1,ALL Challenge,5,32
2,Adam Challenge,2,10
3,BioMediTech,18,38
4,Blood Cell,27,64
5,BreakHis,21,52
6,COVIDx CXR-4,16,34
7,CRC100k,54,84
8,Chest CT Scan,26,54
9,Chest X-Ray PA,26,55


In [68]:
def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            json_line = json.dumps(item, ensure_ascii=False)
            f.write(json_line + "\n")


df_split_summary_.to_csv(os.path.join(
    root, "data_split_summary.csv"), index=False)

save_jsonl(train_json_l, os.path.join(root, "omnimedvqa_train_index.jsonl"))
save_jsonl(test_json_l, os.path.join(root, "omnimedvqa_test_index.jsonl"))

In [ ]:
{
    "question_id": "RadImageNet_33584",
    "dataset": "RadImageNet",
    "image_path": "Images/RadImageNet/chondral_abnormality/foot051933.png",
    "access_type": "Open-access",
}